In [2]:
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



# 1 Define project files and final model parameters


BASE_DIR = Path.cwd()

SIMULATION_PATH = (
    BASE_DIR
    / "phase10_synthetic_learner_environment_rerun.csv"
)

POLICY_PATH = (
    BASE_DIR
    / "phase14_qlearning_policy_rerun.csv"
)

# Use the paired baseline evaluation already completed in Phase 15
BASELINE_SOURCE_PATH = (
    BASE_DIR
    / "phase15_policy_comparison_rerun.csv"
)

FINAL_POLICY_PATH = (
    BASE_DIR
    / "phase17_final_haptic_policy_rerun.csv"
)

SUMMARY_PATH = (
    BASE_DIR
    / "phase17_final_model_summary_rerun.csv"
)

EPISODE_PATH = (
    BASE_DIR
    / "phase17_final_episode_results_rerun.csv"
)

DECISION_PATH = (
    BASE_DIR
    / "phase17_final_decision_results_rerun.csv"
)

BASELINE_PATH = (
    BASE_DIR
    / "phase17_final_baseline_comparison_rerun.csv"
)

PROFILE_PATH = (
    BASE_DIR
    / "phase17_final_profile_results_rerun.csv"
)

ACTION_PATH = (
    BASE_DIR
    / "phase17_final_action_distribution_rerun.csv"
)

REWARD_PLOT = (
    BASE_DIR
    / "phase17_reward_distribution_rerun.png"
)

ACCURACY_PLOT = (
    BASE_DIR
    / "phase17_accuracy_distribution_rerun.png"
)

ACTION_PLOT = (
    BASE_DIR
    / "phase17_action_distribution_rerun.png"
)

BASELINE_REWARD_PLOT = (
    BASE_DIR
    / "phase17_baseline_reward_comparison_rerun.png"
)

BASELINE_ACCURACY_PLOT = (
    BASE_DIR
    / "phase17_baseline_accuracy_comparison_rerun.png"
)


HAPTIC_ACTIONS = {
    0: "Low",
    1: "Medium",
    2: "High"
}

HAPTIC_COST = {
    0: 0.00,
    1: 0.03,
    2: 0.08
}


print(
    "Simulation file exists:",
    SIMULATION_PATH.exists()
)

print(
    "Policy file exists:",
    POLICY_PATH.exists()
)

print(
    "Phase 15 baseline file exists:",
    BASELINE_SOURCE_PATH.exists()
)


if not SIMULATION_PATH.exists():

    raise FileNotFoundError(
        f"Phase 10 rerun file not found: "
        f"{SIMULATION_PATH}"
    )


if not POLICY_PATH.exists():

    raise FileNotFoundError(
        f"Phase 14 rerun policy not found: "
        f"{POLICY_PATH}"
    )


if not BASELINE_SOURCE_PATH.exists():

    raise FileNotFoundError(
        f"Phase 15 baseline file not found: "
        f"{BASELINE_SOURCE_PATH}"
    )



# 2 Load the synthetic learner environment


simulation_df = pd.read_csv(
    SIMULATION_PATH
)

required_columns = [
    "Difficulty_D",
    "Cognitive_Effort_C",
    "D_level",
    "C_level",
    "Learner_Profile",
    "Learner_Ability",
    "Performance_P"
]

missing_columns = [
    column
    for column in required_columns
    if column not in simulation_df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + ", ".join(
            missing_columns
        )
    )


print(
    "\nSimulation shape:",
    simulation_df.shape
)

print(
    "\nMissing simulation values:"
)

print(
    simulation_df[
        required_columns
    ]
    .isna()
    .sum()
)


if (
    simulation_df[
        required_columns
    ]
    .isna()
    .any()
    .any()
):

    raise ValueError(
        "Missing values found "
        "in the simulation data."
    )



# 3 Load and freeze the final Phase 14 Q-learning policy


policy_df = pd.read_csv(
    POLICY_PATH
)

FINAL_POLICY = {}

for _, row in policy_df.iterrows():

    state = (
        int(
            row["D_level"]
        ),
        int(
            row["C_level"]
        ),
        int(
            row["P_level"]
        )
    )

    FINAL_POLICY[
        state
    ] = int(
        row[
            "Haptic_Action"
        ]
    )


if len(
    FINAL_POLICY
) != 27:

    raise ValueError(
        "The final policy should contain "
        "exactly 27 states."
    )


print(
    "\nFinal policy states:",
    len(
        FINAL_POLICY
    )
)

print(
    "Policy action counts:",
    Counter(
        FINAL_POLICY.values()
    )
)



# 4 Define learner performance levels and full RL state


def performance_level(
    performance
):

    if performance < 1 / 3:

        return 0

    elif performance < 2 / 3:

        return 1

    else:

        return 2


def state_full(
    row,
    performance
):

    return (
        int(
            row[
                "D_level"
            ]
        ),

        int(
            row[
                "C_level"
            ]
        ),

        performance_level(
            performance
        )
    )



# 5 Define the final simulated learner environment


def haptic_effect(
    action,
    difficulty,
    cognitive_effort,
    performance
):

    need = (
        0.4 * difficulty
        + 0.4 * cognitive_effort
        + 0.2 * (
            1 - performance
        )
    )

    if action == 0:

        return 0.00

    elif action == 1:

        return (
            0.15 * need
        )

    elif action == 2:

        return (
            0.30 * need
        )

    else:

        raise ValueError(
            "Action must be 0, 1, or 2."
        )


def probability_correct(
    ability,
    difficulty,
    cognitive_effort,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):

    score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    return (
        1
        / (
            1
            + np.exp(
                -score
            )
        )
    )


def probability_correct_with_haptics(
    ability,
    difficulty,
    cognitive_effort,
    performance,
    action,
    alpha=4.0,
    beta=3.0,
    gamma=2.0
):

    base_score = (
        alpha * ability
        - beta * difficulty
        - gamma * cognitive_effort
    )

    effect = haptic_effect(
        action=action,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort,
        performance=performance
    )

    final_score = (
        base_score
        + effect
    )

    return (
        1
        / (
            1
            + np.exp(
                -final_score
            )
        )
    )


def calculate_reward(
    correct,
    p_before,
    p_after,
    action
):

    success_reward = (
        1.0
        if correct == 1
        else -0.5
    )

    improvement = (
        p_after
        - p_before
    )

    reward = (
        success_reward
        + 2.0 * improvement
        - HAPTIC_COST[
            action
        ]
    )

    return reward


def environment_step(
    row,
    performance,
    action,
    rng
):

    ability = float(
        row[
            "Learner_Ability"
        ]
    )

    difficulty = float(
        row[
            "Difficulty_D"
        ]
    )

    cognitive_effort = float(
        row[
            "Cognitive_Effort_C"
        ]
    )

    p_before = probability_correct(
        ability=ability,
        difficulty=difficulty,
        cognitive_effort=cognitive_effort
    )

    p_after = (
        probability_correct_with_haptics(
            ability=ability,
            difficulty=difficulty,
            cognitive_effort=cognitive_effort,
            performance=performance,
            action=action
        )
    )

    p_after = np.clip(
        p_after,
        0.0,
        1.0
    )

    correct = rng.binomial(
        n=1,
        p=p_after
    )

    reward = calculate_reward(
        correct=correct,
        p_before=p_before,
        p_after=p_after,
        action=action
    )

    new_performance = (
        0.8 * performance
        + 0.2 * correct
    )

    new_performance = np.clip(
        new_performance,
        0.0,
        1.0
    )

    return (
        new_performance,
        reward,
        correct,
        p_before,
        p_after
    )



# 6 Export the frozen final haptic policy

LEVEL_NAMES = {
    0: "Low",
    1: "Medium",
    2: "High"
}

final_policy_rows = []

for state, action in sorted(
    FINAL_POLICY.items()
):

    d, c, p = (
        state
    )

    final_policy_rows.append(
        {
            "D_level":
                d,

            "D_label":
                LEVEL_NAMES[
                    d
                ],

            "C_level":
                c,

            "C_label":
                LEVEL_NAMES[
                    c
                ],

            "P_level":
                p,

            "P_label":
                LEVEL_NAMES[
                    p
                ],

            "Haptic_Action":
                action,

            "Haptic_Label":
                HAPTIC_ACTIONS[
                    action
                ]
        }
    )


final_policy_df = pd.DataFrame(
    final_policy_rows
)

final_policy_df.to_csv(
    FINAL_POLICY_PATH,
    index=False
)


print(
    "\nFinal policy action counts:"
)

print(
    final_policy_df[
        "Haptic_Label"
    ]
    .value_counts()
)



# 7 Define the final adaptive-policy evaluator


def evaluate_adaptive_policy(
    n_episodes=500,
    steps_per_episode=50,
    seed=2026,
    profile=None
):

    rng = np.random.default_rng(
        seed
    )

    if profile is None:

        source_df = (
            simulation_df
        )

    else:

        source_df = (
            simulation_df[
                simulation_df[
                    "Learner_Profile"
                ]
                == profile
            ]
            .copy()
        )

    if len(
        source_df
    ) == 0:

        raise ValueError(
            f"No rows found for profile: "
            f"{profile}"
        )

    episode_rows = []

    decision_rows = []


    for episode in range(
        n_episodes
    ):

        row = source_df.iloc[
            rng.integers(
                0,
                len(
                    source_df
                )
            )
        ]

        learner_profile = (
            row[
                "Learner_Profile"
            ]
        )

        performance = float(
            row[
                "Performance_P"
            ]
        )

        total_reward = 0.0

        total_correct = 0


        for step in range(
            steps_per_episode
        ):

            state = state_full(
                row,
                performance
            )

            action = int(
                FINAL_POLICY[
                    state
                ]
            )

            performance_before = (
                performance
            )

            (
                new_performance,
                reward,
                correct,
                p_before,
                p_after
            ) = environment_step(
                row=row,
                performance=performance,
                action=action,
                rng=rng
            )

            decision_rows.append(
                {
                    "Episode":
                        episode + 1,

                    "Step":
                        step + 1,

                    "Learner_Profile":
                        learner_profile,

                    "D_level":
                        state[0],

                    "C_level":
                        state[1],

                    "P_level":
                        state[2],

                    "Action":
                        action,

                    "Haptic_Label":
                        HAPTIC_ACTIONS[
                            action
                        ],

                    "Reward":
                        reward,

                    "Correct":
                        correct,

                    "Performance_Before":
                        performance_before,

                    "Performance_After":
                        new_performance,

                    "P_correct_before":
                        p_before,

                    "P_correct_after":
                        p_after
                }
            )

            total_reward += (
                reward
            )

            total_correct += (
                correct
            )

            performance = (
                new_performance
            )

            # Continue with another passage
            # from the same learner profile
            learner_rows = (
                simulation_df[
                    simulation_df[
                        "Learner_Profile"
                    ]
                    == learner_profile
                ]
            )

            row = learner_rows.iloc[
                rng.integers(
                    0,
                    len(
                        learner_rows
                    )
                )
            ]


        episode_rows.append(
            {
                "Episode":
                    episode + 1,

                "Learner_Profile":
                    learner_profile,

                "Mean_Reward":
                    total_reward
                    / steps_per_episode,

                "Accuracy":
                    total_correct
                    / steps_per_episode
            }
        )


    return (
        pd.DataFrame(
            episode_rows
        ),

        pd.DataFrame(
            decision_rows
        )
    )



# 8 Run the final 500-episode adaptive-policy evaluation


FINAL_EVALUATION_EPISODES = (
    500
)

STEPS_PER_EPISODE = (
    50
)

FINAL_EVALUATION_SEED = (
    2026
)


(
    final_episode_df,
    final_decision_df
) = evaluate_adaptive_policy(
    n_episodes=FINAL_EVALUATION_EPISODES,
    steps_per_episode=STEPS_PER_EPISODE,
    seed=FINAL_EVALUATION_SEED
)


print(
    "\nFinal evaluation episodes:",
    len(
        final_episode_df
    )
)

print(
    "Final evaluation decisions:",
    len(
        final_decision_df
    )
)



# 9 Summarise final adaptive-policy performance


mean_performance_change = (
    final_decision_df[
        "Performance_After"
    ]
    - final_decision_df[
        "Performance_Before"
    ]
).mean()


mean_probability_change = (
    final_decision_df[
        "P_correct_after"
    ]
    - final_decision_df[
        "P_correct_before"
    ]
).mean()


final_summary_df = pd.DataFrame(
    {
        "Metric": [
            "Mean Reward",
            "Reward SD",
            "Mean Accuracy",
            "Accuracy SD",
            "Low Action Proportion",
            "Medium Action Proportion",
            "High Action Proportion",
            "Mean Performance Change",
            "Mean P(correct) Change"
        ],

        "Value": [
            final_episode_df[
                "Mean_Reward"
            ].mean(),

            final_episode_df[
                "Mean_Reward"
            ].std(
                ddof=1
            ),

            final_episode_df[
                "Accuracy"
            ].mean(),

            final_episode_df[
                "Accuracy"
            ].std(
                ddof=1
            ),

            (
                final_decision_df[
                    "Action"
                ]
                == 0
            ).mean(),

            (
                final_decision_df[
                    "Action"
                ]
                == 1
            ).mean(),

            (
                final_decision_df[
                    "Action"
                ]
                == 2
            ).mean(),

            mean_performance_change,

            mean_probability_change
        ]
    }
)


print(
    "\nFINAL MODEL SUMMARY"
)

print(
    "-------------------"
)

print(
    final_summary_df.round(
        5
    )
)


final_summary_df.to_csv(
    SUMMARY_PATH,
    index=False
)

final_episode_df.to_csv(
    EPISODE_PATH,
    index=False
)

final_decision_df.to_csv(
    DECISION_PATH,
    index=False
)



# 10 Calculate the final haptic-action distribution


action_counts = (
    final_decision_df[
        "Haptic_Label"
    ]
    .value_counts()
    .reindex(
        [
            "Low",
            "Medium",
            "High"
        ],
        fill_value=0
    )
)


action_distribution_df = pd.DataFrame(
    {
        "Haptic_Action":
            action_counts.index,

        "Count":
            action_counts.values,

        "Proportion":
            (
                action_counts.values
                / action_counts.sum()
            )
    }
)


print(
    "\nACTION DISTRIBUTION"
)

print(
    "-------------------"
)

print(
    action_distribution_df.round(
        4
    )
)


action_distribution_df.to_csv(
    ACTION_PATH,
    index=False
)



# 11 Load the paired Phase 15 baseline comparison


baseline_df = pd.read_csv(
    BASELINE_SOURCE_PATH
)

expected_policies = {
    "Low",
    "Medium",
    "High",
    "Random",
    "Q-learning"
}

actual_policies = set(
    baseline_df[
        "Policy"
    ]
)

if actual_policies != expected_policies:

    raise ValueError(
        "Phase 15 baseline file does not "
        "contain the expected five policies."
    )


# Save a Phase 17 copy of the validated
# Phase 15 paired comparison
baseline_df.to_csv(
    BASELINE_PATH,
    index=False
)


print(
    "\nBASELINE COMPARISON "
    "(from paired Phase 15 evaluation)"
)

print(
    "------------------------------------------"
)

print(
    baseline_df.round(
        4
    )
)



# 12 Evaluate the final policy by learner profile


profiles = sorted(
    simulation_df[
        "Learner_Profile"
    ]
    .dropna()
    .unique()
)

profile_rows = []

for profile in profiles:

    (
        profile_episode_df,
        profile_decision_df
    ) = evaluate_adaptive_policy(
        n_episodes=300,
        steps_per_episode=50,
        seed=2026,
        profile=profile
    )

    profile_rows.append(
        {
            "Learner_Profile":
                profile,

            "Mean_Reward":
                profile_episode_df[
                    "Mean_Reward"
                ].mean(),

            "SD_Reward":
                profile_episode_df[
                    "Mean_Reward"
                ].std(
                    ddof=1
                ),

            "Mean_Accuracy":
                profile_episode_df[
                    "Accuracy"
                ].mean(),

            "SD_Accuracy":
                profile_episode_df[
                    "Accuracy"
                ].std(
                    ddof=1
                ),

            "Low_Prop":
                (
                    profile_decision_df[
                        "Action"
                    ]
                    == 0
                ).mean(),

            "Medium_Prop":
                (
                    profile_decision_df[
                        "Action"
                    ]
                    == 1
                ).mean(),

            "High_Prop":
                (
                    profile_decision_df[
                        "Action"
                    ]
                    == 2
                ).mean(),

            "Mean_Performance_Change":
                (
                    profile_decision_df[
                        "Performance_After"
                    ]
                    - profile_decision_df[
                        "Performance_Before"
                    ]
                ).mean(),

            "Mean_Probability_Change":
                (
                    profile_decision_df[
                        "P_correct_after"
                    ]
                    - profile_decision_df[
                        "P_correct_before"
                    ]
                ).mean()
        }
    )


profile_final_df = pd.DataFrame(
    profile_rows
)


print(
    "\nRESULTS BY LEARNER PROFILE"
)

print(
    "--------------------------"
)

print(
    profile_final_df.round(
        4
    )
)


profile_final_df.to_csv(
    PROFILE_PATH,
    index=False
)



# 13 Check the final policy across all 27 states

print(
    "\nFINAL 27-STATE POLICY"
)

print(
    "---------------------"
)

print(
    final_policy_df[
        [
            "D_label",
            "C_label",
            "P_label",
            "Haptic_Label"
        ]
    ]
)


print(
    "\nNumber of policy states:",
    len(
        final_policy_df
    )
)


print(
    "\nPolicy-level action counts:"
)

print(
    final_policy_df[
        "Haptic_Label"
    ]
    .value_counts()
)



# 14 Check state visitation during final evaluation


visited_states = (
    final_decision_df[
        [
            "D_level",
            "C_level",
            "P_level"
        ]
    ]
    .drop_duplicates()
)


print(
    "\nStates visited during final evaluation:",
    len(
        visited_states
    ),
    "/ 27"
)


state_visit_counts = (
    final_decision_df
    .groupby(
        [
            "D_level",
            "C_level",
            "P_level"
        ]
    )
    .size()
    .reset_index(
        name="Count"
    )
)


print(
    "\nMost frequently visited states:"
)

print(
    state_visit_counts
    .sort_values(
        "Count",
        ascending=False
    )
    .head(10)
)



# 15 Check final evaluation data integrity


decision_columns_to_check = [
    "Reward",
    "Correct",
    "Performance_Before",
    "Performance_After",
    "P_correct_before",
    "P_correct_after"
]


print(
    "\nMissing values in final decision data:"
)

print(
    final_decision_df[
        decision_columns_to_check
    ]
    .isna()
    .sum()
)


if (
    final_decision_df[
        decision_columns_to_check
    ]
    .isna()
    .any()
    .any()
):

    raise ValueError(
        "Missing values found in final "
        "decision-level results."
    )


if len(
    final_episode_df
) != FINAL_EVALUATION_EPISODES:

    raise ValueError(
        "Unexpected number of final "
        "evaluation episodes."
    )


expected_decisions = (
    FINAL_EVALUATION_EPISODES
    * STEPS_PER_EPISODE
)


if len(
    final_decision_df
) != expected_decisions:

    raise ValueError(
        "Unexpected number of final "
        "evaluation decisions."
    )


print(
    "\nFinal evaluation integrity check passed."
)

print(
    "Episodes:",
    len(
        final_episode_df
    )
)

print(
    "Decisions:",
    len(
        final_decision_df
    )
)



# 16 Plot the final reward distribution

plt.figure(
    figsize=(8, 5)
)

plt.hist(
    final_episode_df[
        "Mean_Reward"
    ],
    bins=25
)

plt.xlabel(
    "Mean Reward per Episode"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Final Q-learning Reward Distribution"
)

plt.tight_layout()

plt.savefig(
    REWARD_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 17 Plot the final simulated accuracy distribution


plt.figure(
    figsize=(8, 5)
)

plt.hist(
    final_episode_df[
        "Accuracy"
    ],
    bins=25
)

plt.xlabel(
    "Episode Accuracy"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Final Simulated Accuracy Distribution"
)

plt.tight_layout()

plt.savefig(
    ACCURACY_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 18 Plot the final haptic-action distribution


plt.figure(
    figsize=(7, 5)
)

plt.bar(
    action_distribution_df[
        "Haptic_Action"
    ],
    action_distribution_df[
        "Proportion"
    ]
)

plt.xlabel(
    "Haptic Action"
)

plt.ylabel(
    "Proportion of Decisions"
)

plt.title(
    "Final Haptic Action Distribution"
)

plt.tight_layout()

plt.savefig(
    ACTION_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 19 Plot the paired Phase 15 baseline reward comparison


plt.figure(
    figsize=(8, 5)
)

plt.bar(
    baseline_df[
        "Policy"
    ],
    baseline_df[
        "Mean_Reward"
    ]
)

plt.xlabel(
    "Policy"
)

plt.ylabel(
    "Mean Reward per Step"
)

plt.title(
    "Final Policy Comparison: Mean Reward"
)

plt.tight_layout()

plt.savefig(
    BASELINE_REWARD_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 20 Plot the paired Phase 15 baseline accuracy comparison

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    baseline_df[
        "Policy"
    ],
    baseline_df[
        "Mean_Accuracy"
    ]
)

plt.xlabel(
    "Policy"
)

plt.ylabel(
    "Mean Simulated Accuracy"
)

plt.title(
    "Final Policy Comparison: Accuracy"
)

plt.tight_layout()

plt.savefig(
    BASELINE_ACCURACY_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 21 Print final output summary


print(
    "\nFINAL PHASE 17 SUMMARY"
)

print(
    "----------------------"
)


print(
    "\nFrozen policy:"
)

print(
    "States:",
    len(
        FINAL_POLICY
    )
)

print(
    "Actions:",
    Counter(
        FINAL_POLICY.values()
    )
)


print(
    "\nFinal adaptive evaluation:"
)

print(
    "Mean reward:",
    round(
        final_episode_df[
            "Mean_Reward"
        ].mean(),
        4
    )
)

print(
    "Reward SD:",
    round(
        final_episode_df[
            "Mean_Reward"
        ].std(
            ddof=1
        ),
        4
    )
)

print(
    "Mean accuracy:",
    round(
        final_episode_df[
            "Accuracy"
        ].mean(),
        4
    )
)

print(
    "Accuracy SD:",
    round(
        final_episode_df[
            "Accuracy"
        ].std(
            ddof=1
        ),
        4
    )
)


print(
    "\nDecision-level action proportions:"
)

for action in [
    0,
    1,
    2
]:

    proportion = (
        final_decision_df[
            "Action"
        ]
        == action
    ).mean()

    print(
        HAPTIC_ACTIONS[
            action
        ],
        ":",
        round(
            proportion,
            4
        )
    )


print(
    "\nMean P(correct) change:",
    round(
        mean_probability_change,
        5
    )
)

print(
    "Mean performance change:",
    round(
        mean_performance_change,
        5
    )
)



# 22 Confirm all saved Phase 17 outputs


print(
    "\nPHASE 17 COMPLETE"
)

print(
    "Saved final policy:",
    FINAL_POLICY_PATH
)

print(
    "Saved final summary:",
    SUMMARY_PATH
)

print(
    "Saved episode results:",
    EPISODE_PATH
)

print(
    "Saved decision results:",
    DECISION_PATH
)

print(
    "Saved action distribution:",
    ACTION_PATH
)

print(
    "Saved baseline comparison:",
    BASELINE_PATH
)

print(
    "Saved learner-profile results:",
    PROFILE_PATH
)

print(
    "Saved reward distribution plot:",
    REWARD_PLOT
)

print(
    "Saved accuracy distribution plot:",
    ACCURACY_PLOT
)

print(
    "Saved action distribution plot:",
    ACTION_PLOT
)

print(
    "Saved baseline reward plot:",
    BASELINE_REWARD_PLOT
)

print(
    "Saved baseline accuracy plot:",
    BASELINE_ACCURACY_PLOT
)

Simulation file exists: True
Policy file exists: True
Phase 15 baseline file exists: True

Simulation shape: (2127, 37)

Missing simulation values:
Difficulty_D          0
Cognitive_Effort_C    0
D_level               0
C_level               0
Learner_Profile       0
Learner_Ability       0
Performance_P         0
dtype: int64

Final policy states: 27
Policy action counts: Counter({0: 10, 2: 9, 1: 8})

Final policy action counts:
Haptic_Label
Low       10
High       9
Medium     8
Name: count, dtype: int64

Final evaluation episodes: 500
Final evaluation decisions: 25000

FINAL MODEL SUMMARY
-------------------
                     Metric    Value
0               Mean Reward  0.24699
1                 Reward SD  0.26532
2             Mean Accuracy  0.50148
3               Accuracy SD  0.17595
4     Low Action Proportion  0.39656
5  Medium Action Proportion  0.31240
6    High Action Proportion  0.29104
7   Mean Performance Change  0.00029
8    Mean P(correct) Change  0.01371

ACTION DIS